# Tiny GPT From Scratch v2 (Jetson CUDA + Mac MPS)

This notebook trains a small **character-level GPT** language model on Tiny Shakespeare, then lets you load a saved checkpoint and chat quickly.

What to expect:
- This is a **base next-token model**, not an instruction-tuned chat assistant.
- It can still feel chat-like if you prompt with structure, e.g. `User: ...\nAssistant:`.
- Quality improves with more/broader text and better tokenization (see Next Steps).

In [ ]:
import torch

if torch.backends.mps.is_available():
    device = "mps"
elif torch.cuda.is_available():
    device = "cuda"
else:
    device = "cpu"

print("torch version:", torch.__version__)
print("mps available:", torch.backends.mps.is_available())
print("cuda available:", torch.cuda.is_available())
print("using device:", device)

if device == "cuda":
    print("cuda device:", torch.cuda.get_device_name(0))

try:
    torch.set_float32_matmul_precision("high")
    print("set_float32_matmul_precision('high') enabled")
except Exception as e:
    print("matmul precision setting skipped:", e)

In [ ]:
import subprocess
from pathlib import Path

DATA_PATH = Path("input.txt")
URL = "https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt"

if not DATA_PATH.exists():
    print("input.txt not found. Downloading Tiny Shakespeare...")
    try:
        subprocess.run(["wget", "-O", str(DATA_PATH), URL], check=True)
        print("Downloaded with wget.")
    except (FileNotFoundError, subprocess.CalledProcessError):
        print("wget unavailable or failed. Falling back to urllib...")
        import urllib.request
        urllib.request.urlretrieve(URL, str(DATA_PATH))
        print("Downloaded with urllib.")
else:
    print("Found existing input.txt")

size_bytes = DATA_PATH.stat().st_size
print(f"file size: {size_bytes:,} bytes")

text_preview = DATA_PATH.read_text(encoding="utf-8")[:300]
print("\nPreview (first ~300 chars):\n")
print(text_preview)

## Char-level Tokenizer

We build a character vocabulary directly from `input.txt`, then map chars to IDs and split into train/validation sets.

In [ ]:
import torch
from pathlib import Path

text = Path("input.txt").read_text(encoding="utf-8")
chars = sorted(list(set(text)))
vocab_size = len(chars)

stoi = {ch: i for i, ch in enumerate(chars)}
itos = {i: ch for i, ch in enumerate(chars)}

encode = lambda s: [stoi[c] for c in s]
decode = lambda ids: "".join(itos[i] for i in ids)

data = torch.tensor(encode(text), dtype=torch.long)

n = int(0.9 * len(data))
train_data = data[:n]
val_data = data[n:]

print("vocab size:", vocab_size)
print("dataset length:", len(data))
print("train length:", len(train_data))
print("val length:", len(val_data))

## Batch Sampler

Defaults are conservative for 8GB-class devices and should also run on MPS. Adjust upward only after stable runs.

In [ ]:
# Resource-aware defaults
block_size = 256
n_layers = 4
n_heads = 4
d_model = 256
dropout = 0.1

batch_size = 16
grad_accum_steps = 4

max_steps = 6000
eval_every = 500
eval_iters = 100

learning_rate = 3e-4
min_lr = 3e-5
warmup_steps = 300
weight_decay = 0.1
beta1, beta2 = 0.9, 0.95
grad_clip = 1.0

amp_enabled = (device == "cuda")


def get_batch(split):
    source = train_data if split == "train" else val_data
    ix = torch.randint(len(source) - block_size - 1, (batch_size,))
    x = torch.stack([source[i:i + block_size] for i in ix])
    y = torch.stack([source[i + 1:i + 1 + block_size] for i in ix])
    return x.to(device), y.to(device)


xb, yb = get_batch("train")
print("x shape:", tuple(xb.shape), "y shape:", tuple(yb.shape))

## Model

In [ ]:
import math
from dataclasses import dataclass
import torch
import torch.nn as nn
import torch.nn.functional as F


class CausalSelfAttention(nn.Module):
    def __init__(self, d_model, n_heads, block_size, dropout):
        super().__init__()
        assert d_model % n_heads == 0
        self.n_heads = n_heads
        self.head_dim = d_model // n_heads

        self.c_attn = nn.Linear(d_model, 3 * d_model)
        self.c_proj = nn.Linear(d_model, d_model)
        self.attn_dropout = nn.Dropout(dropout)
        self.resid_dropout = nn.Dropout(dropout)

        mask = torch.tril(torch.ones(block_size, block_size))
        self.register_buffer("mask", mask.view(1, 1, block_size, block_size))

    def forward(self, x):
        B, T, C = x.shape
        qkv = self.c_attn(x)
        q, k, v = qkv.split(C, dim=2)

        q = q.view(B, T, self.n_heads, self.head_dim).transpose(1, 2)
        k = k.view(B, T, self.n_heads, self.head_dim).transpose(1, 2)
        v = v.view(B, T, self.n_heads, self.head_dim).transpose(1, 2)

        att = (q @ k.transpose(-2, -1)) / math.sqrt(self.head_dim)
        att = att.masked_fill(self.mask[:, :, :T, :T] == 0, float("-inf"))
        att = F.softmax(att, dim=-1)
        att = self.attn_dropout(att)

        y = att @ v
        y = y.transpose(1, 2).contiguous().view(B, T, C)
        y = self.resid_dropout(self.c_proj(y))
        return y


class MLP(nn.Module):
    def __init__(self, d_model, dropout):
        super().__init__()
        self.fc = nn.Linear(d_model, 4 * d_model)
        self.proj = nn.Linear(4 * d_model, d_model)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        x = self.fc(x)
        x = F.gelu(x)
        x = self.proj(x)
        x = self.dropout(x)
        return x


class Block(nn.Module):
    def __init__(self, d_model, n_heads, block_size, dropout):
        super().__init__()
        self.ln1 = nn.LayerNorm(d_model)
        self.attn = CausalSelfAttention(d_model, n_heads, block_size, dropout)
        self.ln2 = nn.LayerNorm(d_model)
        self.mlp = MLP(d_model, dropout)

    def forward(self, x):
        x = x + self.attn(self.ln1(x))
        x = x + self.mlp(self.ln2(x))
        return x


@dataclass
class GPTConfig:
    vocab_size: int
    block_size: int
    n_layers: int
    n_heads: int
    d_model: int
    dropout: float


class GPT(nn.Module):
    def __init__(self, config: GPTConfig):
        super().__init__()
        self.config = config

        self.tok_emb = nn.Embedding(config.vocab_size, config.d_model)
        self.pos_emb = nn.Embedding(config.block_size, config.d_model)
        self.drop = nn.Dropout(config.dropout)

        self.blocks = nn.ModuleList([
            Block(config.d_model, config.n_heads, config.block_size, config.dropout)
            for _ in range(config.n_layers)
        ])
        self.ln_f = nn.LayerNorm(config.d_model)
        self.lm_head = nn.Linear(config.d_model, config.vocab_size, bias=False)

        self.apply(self._init_weights)

    def _init_weights(self, module):
        if isinstance(module, nn.Linear):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)
            if module.bias is not None:
                nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Embedding):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)

    def forward(self, idx, targets=None):
        B, T = idx.shape
        if T > self.config.block_size:
            raise ValueError(f"sequence length {T} > block_size {self.config.block_size}")

        pos = torch.arange(0, T, device=idx.device)
        x = self.tok_emb(idx) + self.pos_emb(pos)[None, :, :]
        x = self.drop(x)

        for block in self.blocks:
            x = block(x)

        x = self.ln_f(x)
        logits = self.lm_head(x)

        loss = None
        if targets is not None:
            loss = F.cross_entropy(
                logits.view(B * T, self.config.vocab_size),
                targets.view(B * T),
            )

        return logits, loss

    @torch.no_grad()
    def generate(self, idx, max_new_tokens, temperature=1.0, top_k=None):
        self.eval()
        for _ in range(max_new_tokens):
            idx_cond = idx[:, -self.config.block_size:]
            logits, _ = self(idx_cond)
            logits = logits[:, -1, :] / max(temperature, 1e-6)

            if top_k is not None:
                v, _ = torch.topk(logits, min(top_k, logits.size(-1)))
                logits[logits < v[:, [-1]]] = -float("inf")

            probs = F.softmax(logits, dim=-1)
            next_idx = torch.multinomial(probs, num_samples=1)
            idx = torch.cat((idx, next_idx), dim=1)

        return idx

In [ ]:
config = GPTConfig(
    vocab_size=vocab_size,
    block_size=block_size,
    n_layers=n_layers,
    n_heads=n_heads,
    d_model=d_model,
    dropout=dropout,
)

model = GPT(config).to(device)

num_params = sum(p.numel() for p in model.parameters())
print(f"parameters: {num_params / 1e6:.2f}M")

dummy_x = torch.randint(0, vocab_size, (2, 16), device=device)
logits, loss = model(dummy_x, dummy_x)
print("logits shape:", tuple(logits.shape))
print("loss shape:", tuple(loss.shape) if hasattr(loss, "shape") else "scalar")
print("loss is scalar:", loss.ndim == 0)

## Training Utilities

In [ ]:
@torch.no_grad()
def estimate_loss(model):
    out = {}
    model.eval()
    for split in ["train", "val"]:
        losses = torch.zeros(eval_iters)
        for k in range(eval_iters):
            x, y = get_batch(split)
            if device in ("cuda", "mps"):
                with torch.autocast(device_type=device, dtype=torch.float16):
                    _, loss = model(x, y)
            else:
                _, loss = model(x, y)
            losses[k] = loss.item()
        out[split] = losses.mean().item()
    model.train()
    return out


def get_lr(step):
    if step < warmup_steps:
        return learning_rate * (step + 1) / warmup_steps
    if step > max_steps:
        return min_lr

    decay_ratio = (step - warmup_steps) / max(1, (max_steps - warmup_steps))
    decay_ratio = min(max(decay_ratio, 0.0), 1.0)
    coeff = 0.5 * (1.0 + math.cos(math.pi * decay_ratio))
    return min_lr + coeff * (learning_rate - min_lr)

## Training Loop

- Uses AdamW, gradient accumulation, and gradient clipping.
- Uses AMP `autocast` on CUDA and MPS.
- Uses `GradScaler` only on CUDA; MPS/CPU use regular backward/step.

### OOM Safety (if memory errors happen)

Reduce in this order:
1. `batch_size`
2. `block_size`
3. `d_model`

Then retry with a fresh kernel.

In [ ]:
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=learning_rate,
    betas=(beta1, beta2),
    weight_decay=weight_decay,
)

use_amp = device in ("cuda", "mps")
use_scaler = device == "cuda"
scaler = torch.cuda.amp.GradScaler() if use_scaler else None

model.train()

for step in range(max_steps + 1):
    lr = get_lr(step)
    for pg in optimizer.param_groups:
        pg["lr"] = lr

    optimizer.zero_grad(set_to_none=True)

    for _ in range(grad_accum_steps):
        xb, yb = get_batch("train")

        if use_amp:
            with torch.autocast(device_type=device, dtype=torch.float16):
                _, loss = model(xb, yb)
                loss = loss / grad_accum_steps
        else:
            _, loss = model(xb, yb)
            loss = loss / grad_accum_steps

        if use_scaler:
            scaler.scale(loss).backward()
        else:
            loss.backward()

    if use_scaler:
        scaler.unscale_(optimizer)

    torch.nn.utils.clip_grad_norm_(model.parameters(), grad_clip)

    if use_scaler:
        scaler.step(optimizer)
        scaler.update()
    else:
        optimizer.step()

    if step % eval_every == 0:
        losses = estimate_loss(model)
        print(f"step {step:5d} | lr {lr:.6f} | train {losses['train']:.4f} | val {losses['val']:.4f}")

        context = torch.zeros((1, 1), dtype=torch.long, device=device)
        sample_ids = model.generate(context, max_new_tokens=300, temperature=0.9, top_k=50)[0].tolist()
        print("sample:\n" + decode(sample_ids))
        print("-" * 80)

In [ ]:
ckpt = {
    "model_state": model.state_dict(),
    "config": {
        "vocab_size": vocab_size,
        "block_size": block_size,
        "n_layers": n_layers,
        "n_heads": n_heads,
        "d_model": d_model,
        "dropout": dropout,
        "chars": chars,
    },
}

torch.save(ckpt, "gpt_char_ckpt.pt")
print("saved checkpoint to gpt_char_ckpt.pt")

## Load + Quick Test

This verifies checkpoint loading and prints a longer sample (600 chars).

In [ ]:
loaded = torch.load("gpt_char_ckpt.pt", map_location=device)
cfg_dict = loaded["config"]

loaded_config = GPTConfig(
    vocab_size=cfg_dict["vocab_size"],
    block_size=cfg_dict["block_size"],
    n_layers=cfg_dict["n_layers"],
    n_heads=cfg_dict["n_heads"],
    d_model=cfg_dict["d_model"],
    dropout=cfg_dict["dropout"],
)

model_loaded = GPT(loaded_config).to(device)
model_loaded.load_state_dict(loaded["model_state"])
model_loaded.eval()

chars_loaded = cfg_dict["chars"]
itos_loaded = {i: ch for i, ch in enumerate(chars_loaded)}

def decode_loaded(ids):
    return "".join(itos_loaded[i] for i in ids)

context = torch.zeros((1, 1), dtype=torch.long, device=device)
with torch.no_grad():
    out = model_loaded.generate(context, max_new_tokens=600, temperature=0.9, top_k=50)

print(decode_loaded(out[0].tolist()))

In [ ]:
import math
import torch
import torch.nn as nn
import torch.nn.functional as F

# ---------- ONE-CELL: Load + Chat ----------
CKPT_PATH = "gpt_char_ckpt.pt"
TEMPERATURE = 0.9
TOP_K = 50
MAX_NEW_TOKENS = 300

if torch.backends.mps.is_available():
    device = "mps"
elif torch.cuda.is_available():
    device = "cuda"
else:
    device = "cpu"

try:
    torch.set_float32_matmul_precision("high")
except Exception:
    pass

print("Torch:", torch.__version__)
print("MPS available:", torch.backends.mps.is_available())
print("CUDA available:", torch.cuda.is_available())
if device == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))
print("Device:", device)

ckpt = torch.load(CKPT_PATH, map_location=device)
cfg = ckpt["config"]

chars = cfg["chars"]
stoi = {ch: i for i, ch in enumerate(chars)}
itos = {i: ch for i, ch in enumerate(chars)}


def encode(s: str):
    return [stoi[c] for c in s]


def decode(ids):
    return "".join(itos[i] for i in ids)


class CausalSelfAttention(nn.Module):
    def __init__(self, d_model, n_heads, block_size, dropout):
        super().__init__()
        assert d_model % n_heads == 0
        self.n_heads = n_heads
        self.head_dim = d_model // n_heads
        self.c_attn = nn.Linear(d_model, 3 * d_model)
        self.c_proj = nn.Linear(d_model, d_model)
        self.attn_dropout = nn.Dropout(dropout)
        self.resid_dropout = nn.Dropout(dropout)
        mask = torch.tril(torch.ones(block_size, block_size))
        self.register_buffer("mask", mask.view(1, 1, block_size, block_size))

    def forward(self, x):
        B, T, C = x.shape
        qkv = self.c_attn(x)
        q, k, v = qkv.split(C, dim=2)
        q = q.view(B, T, self.n_heads, self.head_dim).transpose(1, 2)
        k = k.view(B, T, self.n_heads, self.head_dim).transpose(1, 2)
        v = v.view(B, T, self.n_heads, self.head_dim).transpose(1, 2)
        att = (q @ k.transpose(-2, -1)) / math.sqrt(self.head_dim)
        att = att.masked_fill(self.mask[:, :, :T, :T] == 0, float("-inf"))
        att = F.softmax(att, dim=-1)
        att = self.attn_dropout(att)
        y = att @ v
        y = y.transpose(1, 2).contiguous().view(B, T, C)
        y = self.resid_dropout(self.c_proj(y))
        return y


class MLP(nn.Module):
    def __init__(self, d_model, dropout):
        super().__init__()
        self.fc = nn.Linear(d_model, 4 * d_model)
        self.proj = nn.Linear(4 * d_model, d_model)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        x = self.fc(x)
        x = F.gelu(x)
        x = self.proj(x)
        x = self.dropout(x)
        return x


class Block(nn.Module):
    def __init__(self, d_model, n_heads, block_size, dropout):
        super().__init__()
        self.ln1 = nn.LayerNorm(d_model)
        self.attn = CausalSelfAttention(d_model, n_heads, block_size, dropout)
        self.ln2 = nn.LayerNorm(d_model)
        self.mlp = MLP(d_model, dropout)

    def forward(self, x):
        x = x + self.attn(self.ln1(x))
        x = x + self.mlp(self.ln2(x))
        return x


class GPT(nn.Module):
    def __init__(self, vocab_size, d_model, n_layers, n_heads, block_size, dropout):
        super().__init__()
        self.block_size = block_size
        self.tok_emb = nn.Embedding(vocab_size, d_model)
        self.pos_emb = nn.Embedding(block_size, d_model)
        self.drop = nn.Dropout(dropout)
        self.blocks = nn.ModuleList([
            Block(d_model, n_heads, block_size, dropout) for _ in range(n_layers)
        ])
        self.ln_f = nn.LayerNorm(d_model)
        self.lm_head = nn.Linear(d_model, vocab_size, bias=False)

    def forward(self, idx, targets=None):
        B, T = idx.shape
        if T > self.block_size:
            raise ValueError(f"sequence length {T} > block_size {self.block_size}")
        pos = torch.arange(0, T, device=idx.device)
        x = self.tok_emb(idx) + self.pos_emb(pos)[None, :, :]
        x = self.drop(x)
        for block in self.blocks:
            x = block(x)
        x = self.ln_f(x)
        logits = self.lm_head(x)
        loss = None
        if targets is not None:
            loss = F.cross_entropy(logits.view(B * T, logits.size(-1)), targets.view(B * T))
        return logits, loss

    @torch.no_grad()
    def generate(self, idx, max_new_tokens, temperature=1.0, top_k=None):
        self.eval()
        for _ in range(max_new_tokens):
            idx_cond = idx[:, -self.block_size:]
            logits, _ = self(idx_cond)
            logits = logits[:, -1, :] / max(temperature, 1e-6)
            if top_k is not None:
                v, _ = torch.topk(logits, min(top_k, logits.size(-1)))
                logits[logits < v[:, [-1]]] = -float("inf")
            probs = F.softmax(logits, dim=-1)
            idx_next = torch.multinomial(probs, num_samples=1)
            idx = torch.cat((idx, idx_next), dim=1)
        return idx


model = GPT(
    vocab_size=cfg["vocab_size"],
    d_model=cfg["d_model"],
    n_layers=cfg["n_layers"],
    n_heads=cfg["n_heads"],
    block_size=cfg["block_size"],
    dropout=cfg["dropout"],
).to(device)

model.load_state_dict(ckpt["model_state"])
model.eval()

print(
    f"Loaded {CKPT_PATH} | vocab={cfg['vocab_size']} block={cfg['block_size']} "
    f"d_model={cfg['d_model']} layers={cfg['n_layers']} heads={cfg['n_heads']}"
)
print("Note: this is a base completion model. Prompt style like 'User: ...\nAssistant:' often works better.")


@torch.no_grad()
def complete(prompt, max_new_tokens=MAX_NEW_TOKENS, temperature=TEMPERATURE, top_k=TOP_K):
    ids = torch.tensor([encode(prompt)], dtype=torch.long, device=device)
    out = model.generate(ids, max_new_tokens=max_new_tokens, temperature=temperature, top_k=top_k)
    return decode(out[0].tolist())


def chat_loop():
    print("\nChat started. Type 'exit' to stop.")
    print("Prompt template hint: User: <message>\nAssistant:")
    while True:
        try:
            user_text = input("You: ").strip()
        except EOFError:
            print("\nInput stream closed; exiting chat loop.")
            break

        if user_text.lower() in {"exit", "quit"}:
            print("Bye.")
            break
        if not user_text:
            continue

        prompt = f"User: {user_text}\nAssistant:"
        try:
            out = complete(prompt, max_new_tokens=220, temperature=TEMPERATURE, top_k=TOP_K)
            print("Assistant:", out[len(prompt):].strip(), "\n")
        except KeyError as e:
            print(f"Character not in vocab: {e}. Try simpler ASCII text or retrain with broader data.\n")


chat_loop()

## OPTIONAL: ipywidgets Chat UI

If `ipywidgets` is installed, run the next cell for a small notebook UI chat wrapper around `complete()`.

In [ ]:
try:
    import ipywidgets as widgets
    from IPython.display import display

    if "complete" not in globals():
        print("`complete()` not found. Run the ONE-CELL: Load + Chat cell first.")
    else:
        prompt_box = widgets.Text(
            value="",
            placeholder="Type a message and press Send",
            description="You:",
            layout=widgets.Layout(width="80%"),
        )
        send_btn = widgets.Button(description="Send", button_style="primary")
        out = widgets.Output(layout={"border": "1px solid #bbb", "padding": "8px", "max_height": "320px", "overflow_y": "auto"})

        def on_send(_):
            text = prompt_box.value.strip()
            if not text:
                return
            prompt_box.value = ""
            prompt = f"User: {text}\nAssistant:"
            with out:
                print(f"You: {text}")
                try:
                    ans = complete(prompt, max_new_tokens=220, temperature=0.9, top_k=50)
                    print("Assistant:", ans[len(prompt):].strip())
                except KeyError as e:
                    print(f"Character not in vocab: {e}")
                print("-" * 40)

        send_btn.on_click(on_send)
        display(widgets.VBox([widgets.HBox([prompt_box, send_btn]), out]))
except ImportError:
    print("ipywidgets not installed. Skipping UI. Use chat_loop() from the ONE-CELL chat cell.")

## Next Steps

- Move from char-level to subword tokenization (BPE/SentencePiece) for better efficiency and fewer `KeyError` prompt issues.
- Add instruction/chat tuning on conversation-formatted data so the model follows user intent more reliably.
- Expand corpus quality/size, then tune context length and model scale once memory budget is stable.